In [93]:
import os
import subprocess
import time
from clearml import Task, OutputModel
import getpass
from loguru import logger
import json
import pymysql
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler


# 환경 변수 선언 (Configuration)
work_environ = "company"
phase = os.environ.get("PHASE", "dev")
epochs_cnt = os.environ.get("EPOCHS_CNT", 1)
docker_image = os.environ.get("DOCKER_IMAGE", "172.16.11.236:5000/spire/python:3.12-bullseye")
model_type_str = os.environ.get("MODEL_TYPE_STR", "LSTMAe")
process_id = os.environ.get("PROCESS_ID", "P101")
sub_process_id = os.environ.get("SUB_PROCESS_ID", "1002")
project_name = os.environ.get("PROJECT_NAME", "mes")
if phase == "prod":
    db_conf = json.loads(os.environ.get("DB_CONF", '{"host":"172.16.9.60", "port": 3306, "user": "mlops_detect", "password": "QoS908Z1!", "database": "mes_metric"}'))
else:
    db_conf = json.loads(os.environ.get("DB_CONF", '{"host":"172.18.176.1", "port": 3306, "user": "mlops_detect", "password": "mlops_detect", "database": "mes_metric"}'))    
task_name = f"{process_id}_{sub_process_id}"
BUCKET_NAME = "clearml-data"
output_feature_cols = os.environ.get("OUT_FEATURE_COLS", "internal_temp, humidity, rpm").split(', ')
# ---------------------------------------------------------
# 1. 하이퍼파라미터 설정 (건조기 센서 데이터 최적 Baseline)
# ---------------------------------------------------------
lstmae_params = {
    'seq_len': 60,          # 시퀀스 길이 (예: 1분 단위 수집 시 1시간 분량)
    'n_features': 8,        # 센서 개수 (예: 온도, 습도, 진동)
    'inner_dim': 64,        # LSTM 은닉층 차원
    'bottleneck_dim': 16,   # 압축 차원 (특징 추출 공간)
    'batch_size': 64,       # 배치 크기
    'learning_rate': 0.001, # 초기 학습률
    'dropout': 0.1,         # 시계열 정보 유지를 위한 낮은 드롭아웃
    'epochs': 20            # 학습 에폭 수
}

lstmae_params = os.environ.get("PARAMS", lstmae_params)

if work_environ == "home":
    OBJECT_STORAGE_ENDPOINT = 'http://192.168.0.83:9000'  # Garage 서버 주소 (기본 포트 3900)
    AWS_ACCESS_KEY_ID = '7NFFU2I15W8ASBKI4J7T'
    AWS_ACCESS_SECRET_KEY = 'JGl1E8nCeItLJUvrn48y9FzdhZ+nU1+q95NsmxRH'
    AWS_REGION = 'ap-northeast-2'
else:
    OBJECT_STORAGE_ENDPOINT = 'http://172.16.11.235:9000'  # Garage 서버 주소 (기본 포트 3900)
    AWS_ACCESS_KEY_ID = 'QQY84JF5HCFNC814TE75'
    AWS_ACCESS_SECRET_KEY = 'TOzA6qIU0smDLZhQFS7N8jRUVCB+RbdYoyhtJ3Ma'
    AWS_REGION = 'ap-northeast-2'
    # Clearml 정보
    os.environ['CLEARML_WEB_HOST']='http://172.16.8.168:8080'
    os.environ['CLEARML_API_HOST']='http://172.16.8.168:8008'
    os.environ['CLEARML_FILES_HOST']='http://172.16.8.168:8081'
    os.environ['CLEARML_API_ACCESS_KEY']='NA4TVJLF4MNFPAT8JCSYBOVN0QASF2'
    os.environ['CLEARML_API_SECRET_KEY']='g3ur38Bn2GRwzTGDfcEGJy30iPv0Wx43TYDNhN4S4HjVQn5KX6OIpUbCFwTF2uOCQ3c'
    # YOLO의 자동 ClearML 로깅 비활성화
    os.environ['CLEARML_REGISTER_IGNORE'] = 'True'
    os.environ['CLEARML_SKIP_AUTO_STAGED_VCS'] = '1'
    os.environ['CLEARML_VCS_AUTO_CHECKOUT'] = '0'
    os.environ['CLEARML_VCS_IGNORE_EXTENSIONS'] = '1'
    os.environ['CLEARML_SKIP_VCS_AUTO_CONFIGURATION'] = '1'
    os.environ['CLEARML_SKIP_AUTO_STAGED_VCS'] = '1'
    os.environ['CLEARML_SKIP_VCS_AUTO_CONFIGURATION'] = '1'
    os.environ['CLEARML_SKIP_AUTO_STAGED_VCS'] = '1'
    # Git 정보를 찾지 않도록 설정
    os.environ['CLEARML_SKIP_GIT_CHECK'] = '1'
    # 원격지에서 실행 시 Git clone을 시도하지 않음
    os.environ['CLEARML_FORCE_STORE_DIFF'] = '1'

logger.info(f'phase          : {phase}')
logger.info(f'epochs_cnt     : {epochs_cnt}')
logger.info(f'work_environ   : {work_environ}')
logger.info(f'model_type_str : {model_type_str}')
logger.info(f'process_id     : {process_id}')
logger.info(f'sub_process_id : {sub_process_id}')
logger.info(f'project_name   : {project_name}')
logger.info(f'task_name      : {task_name}')
logger.info(f'BUCKET_NAME    : {BUCKET_NAME}')
logger.info(f'params         : {json.dumps(lstmae_params)}')
logger.info(f'db_conf        : {db_conf}')
logger.info(f'feature_cols   : {feature_cols}')

2026-05-18 00:09:59.090 | INFO     | __main__:<module>:79 - phase          : dev
2026-05-18 00:09:59.091 | INFO     | __main__:<module>:80 - epochs_cnt     : 1
2026-05-18 00:09:59.092 | INFO     | __main__:<module>:81 - work_environ   : company
2026-05-18 00:09:59.092 | INFO     | __main__:<module>:82 - model_type_str : LSTMAe
2026-05-18 00:09:59.093 | INFO     | __main__:<module>:83 - process_id     : P101
2026-05-18 00:09:59.094 | INFO     | __main__:<module>:84 - sub_process_id : 1002
2026-05-18 00:09:59.094 | INFO     | __main__:<module>:85 - project_name   : mes
2026-05-18 00:09:59.095 | INFO     | __main__:<module>:86 - task_name      : P101_1002
2026-05-18 00:09:59.095 | INFO     | __main__:<module>:87 - BUCKET_NAME    : clearml-data
2026-05-18 00:09:59.096 | INFO     | __main__:<module>:88 - params         : {"seq_len": 60, "n_features": 8, "inner_dim": 64, "bottleneck_dim": 16, "batch_size": 64, "learning_rate": 0.001, "dropout": 0.1, "epochs": 20}
2026-05-18 00:09:59.097 | IN

In [94]:
class IllegalFeatureException(Exception):
    """
    IllegalFeatureException
    """
    def __init__(self, msg):
        super().__init__(msg)

from torch.utils.data import Dataset

class MySQLDataSet(Dataset):
    """
    MySQLDataSet
    """
    def __init__(self, window_size: int, db_conf: dict, process_id: str, sub_process_id: str, feature_cols: list, base_scaler=None, limit: int =-1):
        self._window_size = window_size
        self._process_id = process_id
        self._sub_process_id = sub_process_id
        self._table_name = f'{process_id}_{sub_process_id}'
        self._database_name = db_conf['database'] if 'database' in db_conf else 'mes'
        self._db_conf = db_conf
        self._db_handle = self.__connect()
        self._offset_pos = 0
        self._direction = "asc"
        self._db_columns = self.__db_get_columns(table_name=self._table_name)
        self._db_columns_str = ', '.join(self._db_columns)
        self._db_columns_str = self._db_columns_str.replace('timestamp', 'UNIX_TIMESTAMP(timestamp)')
        self._limit = limit
        self._current_rpt_cnt = 0
        self._feature_cols = feature_cols
        self._feature_cols_idx = self.__get_features_cols_num()
        if len(self._feature_cols_idx) != len(self._feature_cols):
            raise IllegalFeatureException(f'Illegal feature. [{self._feature_cols}]')
        self._scaler = self._init_scaler(base_scaler)

    @property
    def db_columns(self) -> list:
        return self._db_columns

    @property
    def feature_cols_idx(self) -> list:
        return self._feature_cols_idx
    
    @property
    def scaler(self):
        return self._scaler

    def __connect(self):
        return pymysql.connect(**self._db_conf)

    def __get_features_cols_num(self):
        pos = []
        if set(self._feature_cols).issubset(set(self._db_columns)):
            for feature in self._feature_cols:
                pos.append(self._db_columns.index(feature))
        return pos

    def _init_scaler(self, base_scaler):
        # 전체를 다 읽지 않고 샘플 데이터를 일부만 읽어 스케일러 기준점(Min/Max)을 잡습니다.
        try:
            with self._db_handle.cursor() as cursor:
                data_num = self._get_count()
                logger.info(f'data_num = {data_num}')
                limit = data_num // 10
                logger.info(f'limit = {limit}')
                query = f"SELECT {self._db_columns_str} FROM {self._table_name} LIMIT {limit if limit >= 1 else data_num % 10}"
                logger.info(f'query = {query}')
                cursor.execute(query)
                rows = cursor.fetchall()
                # df = pd.read_sql_query(query, self._db_handle)
                df = pd.DataFrame(rows, columns=self._db_columns)
                if base_scaler is not None:
                    scaler = base_scaler
                    # 기존 데이터 범위를 유지하면서 새 데이터의 최솟값/최댓값을 반영하여 누적 업데이트
                    scaler.partial_fit(df.values)
                else:
                    scaler = MinMaxScaler()
                    scaler.fit(df.values)
                return scaler
        except Exception as ex:
            logger.error(f"Excetion : {ex}")
            raise ex

    def __db_get_columns(self, table_name: str):
        query = f"""describe {table_name}"""
        columns = []
        try:
            with self._db_handle.cursor() as cursor:
                # DESCRIBE 명령어로 테이블 정보 요청
                cursor.execute(query)
                rows = cursor.fetchall()
                for row in rows:
                    if row[0] not in ("id", "process_id", "sub_process_id"):
                        columns.append(row[0])
        except Exception as ex:
            logger.error(f"Exception: {ex}")
        return columns
    
    def __get_item(self, query: str):
        try:
            with self._db_handle.cursor() as cursor:
                cursor.execute(query)
                rows = cursor.fetchall()
                # logger.info(f'rows = {rows[0]}')
                return pd.DataFrame(rows, columns=self._db_columns)
        except Exception as ex:
            logger.error(f"Exception: {ex}")
        return None

    def _get_count(self):
        query = f'select count(*) from {self._table_name}'
        # logger.info(f'query = {query}')
        count = 0
        try:
            with self._db_handle.cursor() as cursor:
                cursor.execute(query)
                rows = cursor.fetchone()
                count = rows[0]
        except Exception as ex:
            logger.error(f"Exception: {ex}")
        return count

    def __len__(self):
        data_len = self._get_count()
        return data_len - self._window_size

    def __getitem__(self, idx):
        # if self._current_rpt_cnt < self._limit:
        logger.info(f'idx = {idx}')
        query = f'SELECT {self._db_columns_str} FROM {self._table_name} ORDER BY id {self._direction} LIMIT {self._window_size} OFFSET {idx}'
        logger.info(f'query = {query}')
        sequence = self.__get_item(query=query)
        # logger.info(f'sequence = {type(sequence)}, {sequence}')
        logger.info(f'sequence\'s shape = {sequence.shape}')
        if sequence.empty or len(sequence) < self._window_size:
            # 에러 방지용 기본 텐서 반환시에도 정확히 피처 개수를 8로 맞춥니다.
            # return torch.zeros((self._window_size, len(self._db_columns)), dtype=torch.float32)
            return torch.zeros((self._window_size, 8), dtype=torch.float32)
        # 2. 정규화 및 텐서 변환
        scaled_data = self._scaler.transform(sequence.values)
        return torch.tensor(scaled_data, dtype=torch.float32)
        """
        if df.empty or len(df) < self.seq_len:
                # 에러 방지용 기본 텐서 반환시에도 정확히 피처 개수를 3으로 맞춥니다.
                return torch.zeros((self.seq_len, 3), dtype=torch.float32)
            
            # 2. [핵심] 스케일러 적용 전에 데이터프레임에 우리가 원하는 3개 컬럼만 있는지 재확인
            # 만약 다른 컬럼이 섞여있더라도 아래 코드가 3개 컬럼만 필터링합니다.
            target_features = df[['temperature', 'humidity', 'vibration']].values
            
            # 3. 3차원 데이터만 스케일러에 통과
            scaled_data = self.scaler.transform(target_features)
            
            # 최종 반환 텐서의 크기는 (SEQ_LEN, 3)이 됩니다.
            return torch.tensor(scaled_data, dtype=torch.float32)
        """
    def close(self):
        if self._db_handle is not None:
            self._db_handle.close()
            self._db_handle = None

    def __str__(self):
        return f'MySQLDataSet(table_name=\"{self._table_name}\")'
    

In [95]:
run_this=False
if run_this:
    datasets = MySQLDataSet(window_size=lstmae_params['seq_len'], db_conf=db_conf, process_id=process_id, sub_process_id=sub_process_id, feature_cols=feature_cols, limit=2)
    logger.info(f'columns = {datasets.db_columns}')
    logger.info(f'feature_cols_idx = {datasets.feature_cols_idx}')
    for data in datasets:
        if data is None or len(data[0]) <= 0:
            break
        logger.info(f'dataset = {data}')
    datasets.close()

In [96]:
# ---------------------------------------------------------
# 3. LSTM-AutoEncoder 신경망 모델 정의
# ---------------------------------------------------------
class Encoder(nn.Module):
    def __init__(self, seq_len, n_features, inner_dim, bottleneck_dim):
        super(Encoder, self).__init__()
        self.lstm1 = nn.LSTM(n_features, inner_dim, batch_first=True, dropout=0.1)
        self.lstm2 = nn.LSTM(inner_dim, bottleneck_dim, batch_first=True)
        
    def forward(self, x):
        x, _ = self.lstm1(x)
        _, (hidden, _) = self.lstm2(x)
        return hidden.squeeze(0)

class Decoder(nn.Module):
    def __init__(self, seq_len, n_features, inner_dim, bottleneck_dim):
        super(Decoder, self).__init__()
        self.seq_len = seq_len
        self.lstm1 = nn.LSTM(bottleneck_dim, inner_dim, batch_first=True, dropout=0.1)
        self.lstm2 = nn.LSTM(inner_dim, n_features, batch_first=True)
        
    def forward(self, x):
        x = x.unsqueeze(1).repeat(1, self.seq_len, 1)
        x, _ = self.lstm1(x)
        x, _ = self.lstm2(x)
        return x

class LSTMAutoEncoder(nn.Module):
    def __init__(self, seq_len, n_features, inner_dim, bottleneck_dim):
        super(LSTMAutoEncoder, self).__init__()
        self.encoder = Encoder(seq_len, n_features, inner_dim, bottleneck_dim)
        self.decoder = Decoder(seq_len, n_features, inner_dim, bottleneck_dim)
        
    def forward(self, x):
        return self.decoder(self.encoder(x))


In [97]:
# 이전 모델 정보 다운 로드

def download_artifacts_from_clearml(task_id):
    """
    ClearML 서버에서 특정 프로젝트 및 태스크 이름에 맵핑된 
    가중치와 스케일러 파일을 찾아 로컬에 다운로드합니다.
    """
    task = Task.get_task(task_id=task_id)
    
    if task is None:
        raise ValueError(f"지정한 프로젝트 또는 태스크를 ClearML에서 찾을 수 없습니다.")

    # 2) Task에 등록된 artifacts 딕셔너리에서 파일들을 가져옵니다.
    artifacts = task.artifacts
    
    if "lstm_autoencoder_weights" not in artifacts or "minmax_scaler" not in artifacts:
        logger.info("태스크 내에 'lstm_autoencoder_weights' 또는 'minmax_scaler' Artifact가 존재하지 않습니다.")
        return None, None

    # 3) get_local_copy() 함수를 호출하면 ClearML 원격 스토리지(S3 등)에서 
    #    현재 로컬 머신의 임시 캐시 디렉토리로 파일을 자동 다운로드하고 그 절대경로를 반환합니다.
    local_weight_path = artifacts["lstm_autoencoder_weights"].get_local_copy()
    local_scaler_path = artifacts["minmax_scaler"].get_local_copy()
    
    logger.info("ClearML로부터 다운로드 성공!")
    logger.info(f"-> 가중치 로컬 경로: {local_weight_path}")
    logger.info(f"-> 스케일러 로컬 경로: {local_scaler_path}")
    
    return local_weight_path, local_scaler_path


def get_pretrained_weigth(project_name, task_name):
    """
    마지막으로 훈련에 성공한 모델의 가중치 파일 다운 받습니다.
    """
    tasks = Task.query_tasks(
        project_name=project_name,
        task_name=task_name,
        task_filter={
            'status': ['completed'],
            'order_by': ['-created']  # Ensures the most recently updated is first
        }
    )
    logger.info(f'tasks\'len = {len(tasks)}')
    if len(tasks) > 0:
        last_task_id= tasks[0]
        logger.info(f'last_task_id = {last_task_id}')
        local_weight_path, local_scaler_path = download_artifacts_from_clearml(task_id=last_task_id)
    logger.info(f"Weight downloaded to #3: {local_weight_path}")
    logger.info(f"Scaler downloaded to #3: {local_scaler_path}")
    return local_weight_path, local_scaler_path


In [98]:
"""
lstmae_params = {
    'seq_len': 60,          # 시퀀스 길이 (예: 1분 단위 수집 시 1시간 분량)
    'n_features': 3,        # 센서 개수 (예: 온도, 습도, 진동)
    'inner_dim': 64,        # LSTM 은닉층 차원
    'bottleneck_dim': 16,   # 압축 차원 (특징 추출 공간)
    'batch_size': 64,       # 배치 크기
    'learning_rate': 0.001, # 초기 학습률
    'dropout': 0.1,         # 시계열 정보 유지를 위한 낮은 드롭아웃
    'epochs': 20            # 학습 에폭 수
}
"""

# ---------------------------------------------------------
# 3. ClearML 자산 로드 및 점진적 재학습 실행
# ---------------------------------------------------------
def train_and_extract_weights(lstmae_params: dict, pre_weight_filepath_name: str = None, pre_scaler_filepath_name: str = None):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"사용중인 연산 디바이스: {device}")

    base_scaler = None
    if pre_scaler_filepath_name is not None:
        base_scaler = joblib.load(pre_scaler_filepath_name)
    
    # 1) 데이터셋 및 데이터로더 초기화
    datasets = MySQLDataSet(window_size=lstmae_params['seq_len'],
                            db_conf=db_conf,
                            process_id=process_id,
                            sub_process_id=sub_process_id,
                            feature_cols=feature_cols,
                            base_scaler=base_scaler,
                            limit=-1)
    train_loader = DataLoader(datasets, batch_size=lstmae_params['batch_size'], shuffle=True, drop_last=True)
    # 2) 모델 컴포넌트 선언
    model = LSTMAutoEncoder(lstmae_params['seq_len'], lstmae_params['n_features'], lstmae_params['inner_dim'], lstmae_params['bottleneck_dim']).to(device)
    if pre_weight_filepath_name is not None:
        model.load_state_dict(torch.load(pre_weight_filepath_name, map_location=device)) 
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lstmae_params['learning_rate'])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)

    # 3) 모델 트레이닝 루프
    logger.info("--- 훈련 시작 ---")
    model.train()
    try:
        for epoch in range(1, lstmae_params['epochs'] + 1):
            epoch_loss = 0.0
            logger.info(f'epoch = {epoch} / {lstmae_params['epochs'] + 1}')
            for batch in train_loader:
                logger.info(f'batch = {len(batch)}')
            
                inputs = batch.to(device)
            
                optimizer.zero_grad()
                outputs = model(inputs)
            
                loss = criterion(outputs, inputs)
                loss.backward()
                optimizer.step()
            
                epoch_loss += loss.item() * inputs.size(0)
            
            epoch_loss /= len(train_loader.dataset)
            scheduler.step(epoch_loss)
            logger.info(f"Epoch [{epoch}/{lstmae_params['epochs']}] 완료 - Train Loss (MSE): {epoch_loss:.6f}")
    except Exception as ex:
        logger.error(f'Exception : {ex}')
    datasets.close()
    logger.info("--- 훈련 완료 ---")

    # 4) 가중치 및 스케일러 외부 추출 파일 보관
    # 핵심 추출 1: 딥러닝 가중치 텐서 저장
    import os
    if os.path.isdir('weights') == False:
        os.mkdir('weights')
    weight_filepath_name = 'weights/lstm_ae_weights.pth'
    scaler_filepath_name = 'weights/minmax_scaler.pkl'
    torch.save(model.state_dict(), weight_filepath_name)
    # 핵심 추출 2: 추론 단계 전처리를 위한 스케일러 상태 저장
    import joblib
    joblib.dump(datasets.scaler, scaler_filepath_name)

    print("가중치('lstm_ae_weights.pth') 및 스케일러('minmax_scaler.pkl') 파일 추출 완료!")
    return weight_filepath_name, scaler_filepath_name   


In [ ]:
logger.info(f'project_name = {project_name}')
logger.info(f'task_name    = {task_name}')
use_clearml = False
task = None
if use_clearml:
    task = Task.init(project_name=project_name,
                     task_name=task_name,
                     reuse_last_task_id=False,
                     continue_last_task=False,
                     task_type=Task.TaskTypes.training)
    if len(lstmae_params.keys()) > 0:
        task.connect(lstmae_params)
    task.set_parameter('version', '1.0')
    task.set_task_type('training')
    ia_task_initiated = False

    logger.info(f'run in phase = {phase}')
    # task.set_base_docker(docker_image)
    # task.set_container(
    #     # 원격 에이전트가 기반으로 사용할 기본 도커 이미지 (필수 입력)
    #     image=docker_image, 
    #     # 여기에 도커 실행 옵션들을 공백으로 구분한 하나의 문자열로 전달합니다.
    #     arguments="--privileged --device /dev/fuse --cap-add SYS_ADMIN"
    # )

    task.set_base_docker(
        docker_image=docker_image,  # 원격에서 실행할 베이스 이미지
        docker_arguments="--privileged --device /dev/fuse"  # 도커 실행 인자
    )

    if phase == "prod":
        task.execute_remotely(queue_name='services',
                              clone=True,
                              exit_process=False)
        from clearml.config import running_remotely

        if not running_remotely():
            # [로컬 Parent 프로세스 영역]
            print("Parent: 자식 태스크를 원격 에이전트에 전달했습니다.")
            task.close()
            task.delete(delete_artifacts_and_models=True)
            exit()
if use_clearml:
    pre_weight_filepath_name, pre_scaler_filepath_name = get_pretrained_weigth(project_name=project_name,
                                                                               task_name=task_name)
else:
    pre_weight_filepath_name = None
    pre_scaler_filepath_name = None
logger.info(f'pre_weight_filepath_name = {pre_weight_filepath_name}')
logger.info(f'pre_scaler_filepath_name = {pre_scaler_filepath_name}')
weight_filepath_name, scaler_filepath_name = train_and_extract_weights(lstmae_params=lstmae_params,
                                                                       pre_weight_filepath_name=pre_weight_filepath_name,
                                                                       pre_scaler_filepath_name=pre_scaler_filepath_name)
if use_clearml:
    task.close()

2026-05-18 00:10:02.222 | INFO     | __main__:<module>:1 - project_name = mes
2026-05-18 00:10:02.223 | INFO     | __main__:<module>:2 - task_name    = P101_1002
2026-05-18 00:10:02.223 | INFO     | __main__:<module>:49 - pre_weight_filepath_name = None
2026-05-18 00:10:02.224 | INFO     | __main__:<module>:50 - pre_scaler_filepath_name = None
2026-05-18 00:10:02.225 | INFO     | __main__:train_and_extract_weights:19 - 사용중인 연산 디바이스: cpu
2026-05-18 00:10:02.232 | INFO     | __main__:_init_scaler:62 - data_num = 20000
2026-05-18 00:10:02.232 | INFO     | __main__:_init_scaler:64 - limit = 2000
2026-05-18 00:10:02.233 | INFO     | __main__:_init_scaler:66 - query = SELECT UNIX_TIMESTAMP(timestamp), internal_temp, humidity, weight, external_temp, wind_power, rpm, user_action FROM P101_1002 LIMIT 2000
/home/simpson/work/mlops_jupyter_notebooks/.venv/lib/python3.12/site-packages/torch/nn/modules/rnn.py:1009: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-

In [40]:
## 가중치 파일과 스케일 파일 저장
task.upload_artifact(name="lstm_autoencoder_weights", artifact_object=weight_filepath_name)
task.upload_artifact(name="minmax_scaler", artifact_object=scaler_filepath_name)
logger.info(f"모델이 파일로 저장되었습니다. [{weight_filepath_name}]")

AttributeError: 'NoneType' object has no attribute 'upload_artifact'